In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import random
from src.data.static import STATES

random.seed(42)

# Document Size (In inches)
width_min, width_max = 7.0, 8.0
height_min, height_max = 8.5, 11.0

ppi = 300

width = random.uniform(width_min, width_max) * ppi
height = random.uniform(height_min, height_max) * ppi
border_size = random.uniform(0.045, 0.08) * width

state = "STATE OF MISSISSIPPI"
print(f"Width: {width}, Height: {height}, Border: {border_size}")
print(f"Margin: {int(border_size * 0.35)}, Band: {int(border_size * 0.4)}")
print(f"Target State: {state}")

In [5]:
def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    hex_color = hex_color.lstrip("#")
    if len(hex_color) == 3:
        hex_color = "".join(c * 2 for c in hex_color)
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))


def _rgb_to_hex(r: int, g: int, b: int) -> str:
    return f"#{r:02X}{g:02X}{b:02X}"


def lighten(hex_color: str, amount: int) -> str:
    """Lighten a hex color by `amount` percent (0-100)."""
    amount = max(0, min(100, amount))
    r, g, b = _hex_to_rgb(hex_color)
    r = int(r + (255 - r) * (amount / 100))
    g = int(g + (255 - g) * (amount / 100))
    b = int(b + (255 - b) * (amount / 100))
    return _rgb_to_hex(r, g, b)


def darken(hex_color: str, amount: int) -> str:
    """Darken a hex color by `amount` percent (0-100)."""
    amount = max(0, min(100, amount))
    r, g, b = _hex_to_rgb(hex_color)
    factor = 1 - (amount / 100)
    r = int(r * factor)
    g = int(g * factor)
    b = int(b * factor)
    return _rgb_to_hex(r, g, b)


def add_noise(hex_color: str, amount: int = 5) -> str:
    """Add a small random ± variation to each RGB channel (0-255). Makes solid colors look more like printed ink."""
    r, g, b = _hex_to_rgb(hex_color)
    r = max(0, min(255, r + random.randint(-amount, amount)))
    g = max(0, min(255, g + random.randint(-amount, amount)))
    b = max(0, min(255, b + random.randint(-amount, amount)))
    return _rgb_to_hex(r, g, b)


black = "#000000"

palette = {
    "green": ["#2F6F3E", "#3A7A3A", "#4F8A52"],
    "blue": ["#2C5C9A", "#3A6FB0", "#1F4A80"],
    "purple": ["#6C4A8A", "#5B3C77", "#7A5BA3"],
    "teal": ["#2F8A8B", "#1E7A7A", "#3A9FA0"]
}

# Base Colors
guilloche_family = random.choice(list(palette.keys()))
guilloche_color = random.choice(palette[guilloche_family])
background = add_noise(lighten(guilloche_color, random.randint(80, 90)), amount=5)
microtext = darken(guilloche_color, 30)
machine_text = black

In [ ]:
import math

def border_text(
        # general input
        document,
        text_contents,

        # computed math stuff
        width,
        height,
        margin,
        band_thickness,
        font_size_px: int | None = None,
        padding_x: int | None = None,
        padding_y: int | None = None,

        # text gen specific
        orientation: str = "Top",
        color_inverse: bool = False
    ):
    """
    Draw border text on the document. Orientation can be "Top", "Bottom", or "Sides".
    Font size and padding scale with band_thickness and document size if not provided.
    """
    # Scale font using band thickness × doc scale factor
    # Reference: 800px doc = scale 1.0. Larger docs scale up gently (exponent 0.4).
    if font_size_px is None:
        doc_scale = (min(width, height) / 800) ** 0.4
        font_size_px = band_thickness * 0.35 * doc_scale
    if padding_x is None:
        padding_x = font_size_px * 0.4
    if padding_y is None:
        padding_y = font_size_px * 0.2

    # Center of the visible border region (document edge to inner band boundary)
    inner_margin = margin + band_thickness
    band_center = inner_margin / 2

    box_height = min(font_size_px * 1.15 + 2 * padding_y, band_thickness * 0.9)

    # Max box width: leave some breathing room within the border edge
    if orientation == "Sides":
        max_box_width = height * 0.85
    else:
        max_box_width = width * 0.85

    if orientation == "Top":
        cx, cy = width / 2, band_center
        bw = min(len(text_contents) * font_size_px * 0.65 + 2 * padding_x, max_box_width)
        placements = [(cx, cy, bw, text_contents if isinstance(text_contents, str) else str(text_contents), 0)]
    elif orientation == "Bottom":
        cx, cy = width / 2, height - band_center
        bw = min(len(text_contents) * font_size_px * 0.65 + 2 * padding_x, max_box_width)
        placements = [(cx, cy, bw, text_contents if isinstance(text_contents, str) else str(text_contents), 0)]
    elif orientation == "Sides":
        if isinstance(text_contents, (list, tuple)) and len(text_contents) >= 2:
            left_text, right_text = text_contents[0], text_contents[1]
        else:
            s = text_contents if isinstance(text_contents, str) else str(text_contents)
            left_text = right_text = s
        mid_y = height / 2
        l_bw = min(len(left_text) * font_size_px * 0.65 + 2 * padding_x, max_box_width)
        r_bw = min(len(right_text) * font_size_px * 0.65 + 2 * padding_x, max_box_width)
        placements = [
            (band_center, mid_y, l_bw, left_text, -90),
            (width - band_center, mid_y, r_bw, right_text, 90),
        ]
    else:
        raise ValueError("orientation must be 'Top', 'Bottom', or 'Sides'")

    font_family = random.choice(["Palatino Linotype", "Palatino", "serif"])
    for cx, cy, bw, text, rotation in placements:
        # If text is too wide, shrink font to fit
        natural_width = len(text) * font_size_px * 0.65 + 2 * padding_x
        actual_font_size = font_size_px
        if natural_width > max_box_width:
            actual_font_size = font_size_px * (max_box_width / natural_width) * 0.9

        text_attrs = dict(
            text_anchor="middle",
            font_size=f"{actual_font_size:.1f}px",
            font_family=font_family,
            font_weight="bold",
        )
        fg = guilloche_color if color_inverse else background
        bg = background if color_inverse else guilloche_color

        box_x = cx - bw / 2
        box_y = cy - box_height / 2
        text_y = cy + actual_font_size * 0.35

        if rotation != 0:
            g = document.g(transform=f"rotate({rotation}, {cx}, {cy})")
            g.add(document.rect(insert=(box_x, box_y), size=(bw, box_height), fill=bg))
            g.add(document.text(text, insert=(cx, text_y), fill=fg, **text_attrs))
            document.add(g)
        else:
            document.add(document.rect(insert=(box_x, box_y), size=(bw, box_height), fill=bg))
            document.add(document.text(text, insert=(cx, text_y), fill=fg, **text_attrs))

In [ ]:
import svgwrite
from IPython.display import SVG, display
from src.utils.background_generation import BackgroundParams, add_background_to_drawing

document = svgwrite.Drawing(filename="document.svg", size=(width, height))

params = BackgroundParams(
    width=int(width),
    height=int(height),
    border_size=border_size,
    bg_color=background,
    border_color=guilloche_color,
    seed=42,
)

add_background_to_drawing(document, params)

print(f"border_size={params.border_size:.1f}")
print(f"border_pattern={params.border_pattern_type.value}")
print(f"inner_pattern={params.inner_pattern_type.value}")
print(f"height={height}, width={width}")

border_text(document, state, width, height, margin=0, band_thickness=border_size)
border_text(document, state, width, height, margin=0, band_thickness=border_size, orientation="Sides")
border_text(document, state, width, height, margin=0, band_thickness=border_size, orientation="Bottom")

# Show inline in the notebook
svg_xml = document.tostring()
display(SVG(svg_xml))

# Still save to disk if you want the file as well
document.save()